In [58]:
!pip install -qU ollama
!pip install -qU langchain langchain-ollama langchain-community faiss-cpu pypdf pydantic sentence-transformers tiktoken langchain-text-splitters pymupdf chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the so

In [20]:
# Initialize the global LLM and Embeddings — reused throughout the entire notebook
from langchain_ollama import ChatOllama
from langchain_community.embeddings import HuggingFaceEmbeddings

llm = ChatOllama(model='gemma3:1b',)
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

print('LLM ready:', llm.model)
print('Embeddings ready:', embeddings.model_name)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

LLM ready: gemma3:1b
Embeddings ready: all-MiniLM-L6-v2


---
# Section 1 — Models

LangChain works with two types of models, each serving a different purpose.

**What you'll learn:**
- The difference between Chat Models, and Embedding Models
- How to initialize and use `ChatOllama`
- How to generate vector embeddings with `HuggingFaceEmbeddings`

> 🤖 Two flavors of AI brain: one talks back (Chat Model), one turns words into math (Embeddings).

In [21]:
# Ask ChatOllama a simple question
from langchain_core.messages import HumanMessage
txt = 'What is the capital of France? Answer in one sentence.'
response = llm.invoke(txt)
print('Type   :', type(response).__name__)
print('Content:', response.content)

Type   : AIMessage
Content: The Capital of france is Paris.


In [22]:
# Embed a sentence and inspect the vector
sentence = 'I love pizza'
vector = embeddings.embed_query(sentence)

print(f"Sentence : '{sentence}'")
print(f'Dimensions: {len(vector)}')
print(f'First 5  : {[round(v, 4) for v in vector[:5]]}')
print(f'\nEach sentence becomes a list of {len(vector)} numbers — its meaning coordinates.')

Sentence : 'I love pizza'
Dimensions: 384
First 5  : [-0.0989, 0.0336, 0.0103, 0.0532, -0.0805]

Each sentence becomes a list of 384 numbers — its meaning coordinates.


### 🎯 Your Turn!
Pick ANY sentence you like — your name, a fact about your day, your favorite snack — and turn it into "meaning coordinates."

> 💡 Tip: sentences with similar *meaning* (not similar *words*) end up close together in vector space.

In [ ]:
# TODO: replace the blank with YOUR OWN sentence
my_sentence = ____

my_vector = embeddings.embed_query(my_sentence)
print(f"'{my_sentence}'")
print(f'-> {len(my_vector)} dimensions')
print(f'-> first 5 numbers: {[round(v, 4) for v in my_vector[:5]]}')

In [ ]:
# ✅ Solution
my_sentence = "I can't wait for the weekend"
my_vector = embeddings.embed_query(my_sentence)
print(f"'{my_sentence}'")
print(f'-> {len(my_vector)} dimensions')
print(f'-> first 5 numbers: {[round(v, 4) for v in my_vector[:5]]}')

---
# Section 2 — Prompt Templates

Instead of writing raw strings, Prompt Templates let you define **reusable templates** with variables — like a fill-in-the-blank form for your LLM.

**What you'll learn:**
- How to use `PromptTemplate` with input variables
- How to use `ChatPromptTemplate` with system + human messages
- How to use `FewShotPromptTemplate` to teach the LLM by example

### Why Use Prompt Templates?

Imagine you run a travel blog and want to write about many different destinations.  
Instead of writing `"Tell me about Paris"`, `"Tell me about Tokyo"` etc., define **one template**:

```
"Tell me about {destination} in 3 sentences."
```

Then just swap out `destination`. Clean, reusable, maintainable.

> ✍️ Stop copy-pasting prompts like it's 2010 — let the template do the typing for you.

In [23]:
# PromptTemplate: simple string template with variables
from langchain_core.prompts import PromptTemplate

travel_template = PromptTemplate(
    input_variables=['destination', 'seasons'],
    template="I'm planning a trip to {destination} in {seasons}. Give me 3 quick travel tips as bullet points."
)

# .format() fills in the blanks
filled = travel_template.format(destination='Kyoto', seasons='spring')
print('Filled prompt:\n', filled)
print()

response = llm.invoke(filled)
print('LLM Response:\n', response.content)

Filled prompt:
 I'm planning a trip to Kyoto in spring. Give me 3 quick travel tips as bullet points.

LLM Response:
 Okay, here are three quick travel tip suggestions specifically for your April springtime visit to Japan & particularly Kyoto! 

*   **Weather is key:** Spring weather can be fickle! Pack layers - you’ll need lightweight jackets and versatile items like thermal underwear – because temperatures fluctuate.  Don't forget an umbrella; rain showers are quite common in spring.



 *   **Wear Comfortable Clothes & Shoes**: You’ll spend a *lot* of time on your feet, exploring the city – so prioritize comfort!


 *    **Book things in Advance:** Especially for popular temples/gardens and tea-ceremonies, reserving tickets online ahead of time will save you wait times.


Hope these help with planning your trip!

Is there anything else I can assist you with regarding your Kyoto travel plan?


In [24]:

# ChatPromptTemplate: system + human messages for role-based conversations
from langchain_core.prompts import ChatPromptTemplate

chef_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a cheerful Italian chef who gives friendly cooking advice. Keep answers under 3 sentences.'),
    ('human', 'How do I make a perfect {dish}?')
])

messages = chef_prompt.format_messages(dish='carbonara')
print('Message types:', [type(m).__name__ for m in messages])

response = llm.invoke(messages)
print("\nChef's advice:", response.content)

Message types: ['SystemMessage', 'HumanMessage']

Chef's advice: Ciao Bella! It's truly fantastic to help you with this classic dish! First, we need pasta – good quality spaghetti is perfect - then whisk eggs, cheese (pecorine for the best flavor!), and pancetta, until it creates a luscious creaminess. A little salt and pepper are all you really need – simple but sublime!


In [25]:
# FewShotPromptTemplate: teach the LLM with examples before asking your question
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

examples = [
    {'movie': 'The Avengers',  'genre': 'Action'},
    {'movie': 'La La Land',    'genre': 'Romance'},
    {'movie': 'Get Out',       'genre': 'Thriller'},
]

example_prompt = PromptTemplate(
    input_variables=['movie', 'genre'],
    template='Movie: {movie}\nGenre: {genre}'
)



few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix='Classify the genre of the following movies:',
    suffix='Movie: {movie}\nGenre:',
    input_variables=['movie']
)

filled = few_shot_prompt.format(movie='Shrek')
print('Few-shot prompt sent to LLM:\n')
print(filled)

response = llm.invoke(filled)
print('\nPredicted genre:', response.content.strip())

Few-shot prompt sent to LLM:

Classify the genre of the following movies:

Movie: The Avengers
Genre: Action

Movie: La La Land
Genre: Romance

Movie: Get Out
Genre: Thriller

Movie: Shrek
Genre:

Predicted genre: Adventure


### 🎯 Your Turn!
The few-shot prompt only saw 3 genres. Put it to the test — classify a movie of YOUR choice.

> 😏 Bonus laugh: try a genre-bending movie (like "Deadpool" or "Shrek") and see if it gets confused!

In [ ]:
# TODO: replace the blank with a movie title of your choice
filled = few_shot_prompt.format(movie=____)
print(filled)

response = llm.invoke(filled)
print('\nPredicted genre:', response.content.strip())

In [ ]:
# ✅ Solution
filled = few_shot_prompt.format(movie='Deadpool')
print(filled)

response = llm.invoke(filled)
print('\nPredicted genre:', response.content.strip())

---
# Section 3 — Chains (LCEL)

**LCEL** (LangChain Expression Language) lets you connect components using the `|` pipe operator — just like Unix pipes in the terminal.

**What you'll learn:**
- How to build a chain with `prompt | model | parser`
- Sequential chains where output of step 1 feeds into step 2
- `RunnablePassthrough` and `RunnableParallel` for branching
- `.invoke()`, `.batch()`, and `.stream()` for running chains


### What Is LCEL?

The **pipe `|` operator** chains components together:

```python
chain = prompt | model | parser
result = chain.invoke({"input": "..."})
```

Each component receives the output of the previous one and passes its result to the next.  
It's like an assembly line — ingredients go in, a finished dish comes out.


> 🚰 Components piped together like plumbing — data flows in, magic flows out.

In [26]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [27]:
# Simple LCEL chain: prompt | model | parser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


llm = ChatOllama(model='gemma3:1b')
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful book recommender. Be concise.'),
    ('human', "Give me a one-paragraph summary of '{club}' as if recommending it to a friend to support this football club.")
])



chain = prompt | llm

result = chain.invoke({'club': 'Real Madrid'})
print('Club Summary:')
print(result.content)

Club Summary:
Seriously, you NEED to check out ‘Real MAD’! It's a modern take on the classics – incredibly skilled free kicks from Messi and Bale, a huge dose of thrilling attacking play, and genuinely brilliant defense. You won’t just watch it, you’ll *feel* it like it’s going into action. Plus, it's got that incredible history and loyalty vibes everyone loves—it’s easily one of the best football matches to watch these days!


In [28]:
# Sequential chain: output of step 1 becomes input to step 2
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

recipe_name_prompt = ChatPromptTemplate.from_template(
    'Suggest one classic dish name from {cuisine} cuisine. Reply with ONLY the dish name, nothing else.'
)
ingredients_prompt = ChatPromptTemplate.from_template(
    "List the 5 main ingredients for making '{dish}'. Use a simple bullet list."
)

step1 = recipe_name_prompt | llm | StrOutputParser()
step2 = ingredients_prompt | llm | StrOutputParser()

cuisine_input = 'Italian'
dish_name = step1.invoke({'cuisine': cuisine_input})
print(f'Step 1 — Dish for {cuisine_input} cuisine: {dish_name.strip()}')

ingredients = step2.invoke({'dish': dish_name.strip()})
print(f'\nStep 2 — Ingredients for {dish_name.strip()}:\n{ingredients}')

Step 1 — Dish for Italian cuisine: Pasta Carbonara

Step 2 — Ingredients for Pasta Carbonara:
Here are five key ingredients typically used when making pasta carbonara:

*   **Spaghetti Pasta:** The base of the dish!
*   **Guanciale (or pancetta):** Adds incredible flavor and fat needed for emulsification.
*   **Eggs:** Essential to create a creamy sauce with richness.
*   **Pecorino Romano Cheese:**  A salty, aged cheese that brings intense flavour. 
*   **Black Pepper:** Provides crucial spice and aromatics!


In [29]:
# RunnablePassthrough and RunnableParallel
from langchain_core.runnables import RunnableParallel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# RunnableParallel: run two chains simultaneously on the same input
summary_prompt  = ChatPromptTemplate.from_template('In one sentence, what is {topic}?')
fun_fact_prompt = ChatPromptTemplate.from_template('Give one surprising fun fact about {topic}.')

parallel_chain = RunnableParallel(
    summary  = summary_prompt  | llm | StrOutputParser(),
    fun_fact = fun_fact_prompt | llm | StrOutputParser()
)

results = parallel_chain.invoke({'topic': 'chocolate'})
print('Summary  :', results['summary'])
print('Fun Fact :', results['fun_fact'])

Summary  : Chocolate was traditionally made from cacao bean powder mixed with water and then subjected to a process of drying and coloring it to create various flavors.
Fun Fact : Okay, here's a truly surprising and interesting fun fact:

**Chocolate beetles aren’t necessarily dead! In reality, they can survive for months, even up to five years, after turning beige – essentially becoming “living fudge!”** 😋

Isn’t that amazing?! 😊 


Would you like to know more about this unusual phenomenon?


In [30]:
# .invoke(), .batch(), and .stream()
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

simple_chain = (
    ChatPromptTemplate.from_template('Name the capital of {country}. One word only.')
    | llm
    | StrOutputParser()
)

# .invoke() — single input
print('.invoke():')
print(simple_chain.invoke({'country': 'Japan'}))

# .batch() — multiple inputs at once
print('\n.batch():')
results = simple_chain.batch([
    {'country': 'France'},
    {'country': 'Brazil'},
    {'country': 'Australia'}
])
for r in results:
    print(f'  -> {r.strip()}')


.invoke():
Tokyo


.batch():
  -> Paris
  -> São Paulo 

However, that’s not quite right! The correct answer is **Rio**. 😉
  -> Canberra


In [31]:
prompt = ChatPromptTemplate.from_messages([
    ('system','You are an expert in Genarative Ai and the new trends'),
    ('human','{question}')
])
stream_chain = (
    prompt
    | llm
    | StrOutputParser()
)

# .stream() — token by token
print('\n.stream() (streaming token by token):')
for chunk in stream_chain.stream({'question':'What is RAG?'}):
    print(chunk, end='', flush=True)
print()


.stream() (streaming token by token):
Alright, let’s dive into Retrieval Augmented Generation. It's become absolutely critical for how these models like me are now used – really important! Buckle up – I think you probably heard of it but maybe you’re grasping the concept slowly.

**RAG essentially bridges two crucial elements in generative AI systems: retrieval and generation:** 

Here's a breakdown, structured to explain things from different angles, considering the *state* of these technologies:


***

**1. The Big Picture & Why RAG is Emerging**

Traditionally (and still often), generating responses using LLMs – like me – involves essentially feeding in massive training data and letting AI do its thing from scratch. It’s great for fluency and creativity, but it’s incredibly computationally *heavy* and relies heavily on the model already having a broad understanding of a vast dataset. 

RAG solves this fundamental bottleneck by **augmenting generation with external knowledge.**  Thi

### 🎯 Your Turn!
Build your OWN one-line LCEL chain — pick a topic, ask for a fun fact, pipe it through the model and a parser.

> 🧩 Reminder: a chain is just `prompt | llm | parser`. You're basically building a tiny assembly line.

In [ ]:
# TODO: 1) pick the parser that returns a clean string   2) fill in your favorite topic
my_prompt = ChatPromptTemplate.from_template('Give one surprising fun fact about {topic}.')
my_chain  = my_prompt | llm | ____()

print(my_chain.invoke({'topic': ____}))

In [ ]:
# ✅ Solution
my_prompt = ChatPromptTemplate.from_template('Give one surprising fun fact about {topic}.')
my_chain  = my_prompt | llm | StrOutputParser()

print(my_chain.invoke({'topic': 'octopuses'}))

---
# Section 4 — Output Parsers + Pydantic

By default, LLMs return plain text. **Output Parsers** transform that text into exactly the format you need — a list, a JSON object, or a typed Python model.

**What you'll learn:**
- `StrOutputParser` — clean text string
- `JsonOutputParser` — Python dictionary

> 🎯 Stop regex-ing JSON out of LLM text by hand — let the parser do the dirty work.

In [32]:
# Default format of LLM output

llm = ChatOllama(model='gemma3:1b')
prompt = ChatPromptTemplate.from_template('Tell me about {company}.')
chain = prompt | llm
result = chain.invoke({'company': 'Nokia'})
result


AIMessage(content='Okay, let\'s dive into the captivating story and enduring legacy of Nokia! It’s a fascinating history intertwined with design, technology, and even political events that has significantly impacted mobile communications around the globe – arguably even influencing the shape of smartphone design slightly. Here’s an overview of Nokia: \n\n**A Humble Start (1923 - 1987): From Fisherman\'s Tools to Global Telecom Giant:**\n\n* **Origins in Finland:** Nokia’s roots actually trace back to a small trading post founded by Simon and Abram Forsman in Nokia, now part of the greater metropolitan area of Turku, Finnish forests. They were simple woodworkers who started producing fishing tools.\n* **Early Radio Days (1930s-1950s):** The brothers, particularly Sigurd (later known as Alvar), expanded into radio operations, securing contracts to produce and distribute wireless communication devices – including small, long-range radios used for fishing and signaling.  This was a HUGE de

In [33]:
# Using StrOutputParser : returns a clean string

from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model='gemma3:1b')
prompt = ChatPromptTemplate.from_template('Tell me about {company}.')
chain = prompt | llm | StrOutputParser()
result = chain.invoke({'company': 'Nokia'})
result


'Alright, let\'s dive into the story and history of Nokia! Here’s a breakdown covering key information:\n\n**1. A Giant in Mobile Phones (and More!) – Origins & Founding:**\n\n* **Nokia (originally "Sauna-Fabrika"): Born in Finland, 1924.**  Named after Ernom Saunaloppa, who founded the sauna factory in his hometown of Nokia. This initial focus on manufacturing heating appliances proved surprisingly valuable, leading to a crucial pivot into wire technology.\n* **Early Focus: Radio Equipment - The Real Breakthrough:** In the 1930s and 40s, Nokia’s founder, Paululfonicka, recognized the potential of radio communication. They specialized in building radios. This expertise was *essential* for the future shift to phones.\n\n**2.  The Shift to Mobile Phone Technology (and Success):**\n\n* **1973 - The Nokia Telephone:** Recognizing the growing market and offering a robust phone with features like directional speaking—Nokia’s first commercial mobile telephone was introduced.\n* **Success Stor

In [36]:
# JsonOutputParser: returns a Python dictionary
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate

parser = JsonOutputParser()
prompt = ChatPromptTemplate.from_template(
    'Return info about a famous restaurant as JSON with fields: '
    'name, city, cuisine, price_range, must_try_dish.\n'
    'Restaurant: {restaurant}\nReturn ONLY valid JSON, no extra text.DONT USE ```json'
)
chain = prompt | llm | parser

result = chain.invoke({'restaurant': 'Nobu'})
print('Type:', type(result).__name__)
print('Restaurant info:')
print(result)
for k, v in result.items():
    print(f'  {k}: {v}')

Type: dict
Restaurant info:
{'name': 'Nobuy', 'city': 'Honolulu', 'cuisine': 'Japanese', 'price_range': '$$$$', 'must_try_dish': 'Melon.'}
  name: Nobuy
  city: Honolulu
  cuisine: Japanese
  price_range: $$$$
  must_try_dish: Melon.


### 🎯 Your Turn!
Get structured JSON info about YOUR favorite restaurant — and sneak in an extra field the original prompt didn't ask for.

> 🍽️ Hint: just add a new field name (like `rating` or `vibe`) to the field list in the prompt.

In [ ]:
# TODO: 1) add one more field to the list (e.g. 'rating')   2) pick your own restaurant
my_prompt = ChatPromptTemplate.from_template(
    'Return info about a famous restaurant as JSON with fields: '
    'name, city, cuisine, price_range, must_try_dish, ____.\n'
    'Restaurant: {restaurant}\nReturn ONLY valid JSON, no extra text. DONT USE ```json'
)
my_chain = my_prompt | llm | parser
print(my_chain.invoke({'restaurant': ____}))

In [ ]:
# ✅ Solution
my_prompt = ChatPromptTemplate.from_template(
    'Return info about a famous restaurant as JSON with fields: '
    'name, city, cuisine, price_range, must_try_dish, rating.\n'
    'Restaurant: {restaurant}\nReturn ONLY valid JSON, no extra text. DONT USE ```json'
)
my_chain = my_prompt | llm | parser
print(my_chain.invoke({'restaurant': 'In-N-Out Burger'}))

---
# Section 5 — Memory

Without memory, every message to the LLM is treated as a fresh conversation. With memory, the LLM can **remember what you said earlier** — just like talking to a human.

**What you'll learn:**
- `ConversationBufferMemory` — remembers every turn
- `ConversationBufferWindowMemory` — remembers only the last k turns
- `ConversationSummaryMemory` — compresses older turns into a running summary
- How to attach memory to a `ConversationChain`

> 🧠 Without memory, your chatbot has the attention span of a goldfish. Let's fix that.

In [38]:
# ConversationBufferMemory: remembers the full conversation history
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationChain

memory       = ConversationBufferMemory()
conversation = ConversationChain(llm=llm, memory=memory, verbose=False)

turns = [
    "I'm thinking of visiting Japan next autumn for 10 days.",
    'What is the best region to visit during that time?',
    'What traditional food should I try there?',
]

print('=== Planning a Trip to Japan ===\n')
for user_input in turns:
    print(f'You: {user_input}')
    response = conversation.predict(input=user_input)
    print(f'AI : {response}\n')

print('=== What is in Memory ===')
print(memory.buffer)

/tmp/ipykernel_1433/2703299343.py:5: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory       = ConversationBufferMemory()
/tmp/ipykernel_1433/2703299343.py:6: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build a conversational agent with `langchain.agents.create_agent` and persist message history via a LangGraph checkpointer.
  conversation = ConversationChain(llm=llm, memory=memory, verbose=False)


=== Planning a Trip to Japan ===

You: I'm thinking of visiting Japan next autumn for 10 days.
AI : Really exciting! That’s a fantastic planning move – traveling to Japan in Autumn is incredibly beautiful and unique. Did you have particular places in mind? Also, are you considering long-haul or shorter trips during that timeframe? Knowing those details would definitely help me give more tailored advice. 😊

You: What is the best region to visit during that time?
AI : That's a great question! Japan has incredible regional diversity throughout autumn – it’s absolutely not like any other season in Europe or North America! The most popular choice for travelers visiting Japan, and especially in autumn, tends to be the **Osaka region.**

Let me tell you *why* Osaka is such a phenomenal choice. Firstly, its fall foliage – while not as spectacularly vibrant as around Kyoto, it offers stunning colors across several areas including Dazaifu Tenmangu Shrine, Nara Park, and even some parts of the ci

In [39]:
# ConversationBufferWindowMemory: only keeps the last k=2 turns
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_classic.chains import ConversationChain

window_memory = ConversationBufferWindowMemory(k=2)
window_conv   = ConversationChain(llm=llm, memory=window_memory, verbose=False)

topics = [
    'My favorite season is autumn.',
    'I love the color orange.',
    'What is my favorite season?',
    'What color did I mention earlier?',
]

print('=== Window Memory (k=2) Demo ===\n')
for msg in topics:
    print(f'You: {msg}')
    response = window_conv.predict(input=msg)
    print(f'AI : {response}\n')

print('Memory buffer (only last 2 turns stored):')
print(window_memory.buffer)

/tmp/ipykernel_1433/633903210.py:5: LangChainDeprecationWarning: The class `ConversationBufferWindowMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  window_memory = ConversationBufferWindowMemory(k=2)


=== Window Memory (k=2) Demo ===

You: My favorite season is autumn.
AI : Okay! There’s something rather beautiful about autumn, wouldn't you agree? Are you partial to cinnamon spice and pumpkin pie when it comes to autumn colors? 🍂


You: I love the color orange.
AI : Indeed, orange has a wonderful rustic charm! From pumpkins glowing through the leaves to those gorgeous maple trees… it’s a classic for a reason. You know, my analysis suggests that oranges are linked with feelings of warmth and satisfaction; which, naturally, complements the melancholic feeling of autumn. Are you thinking about any particular autumnal activity? 




You: What is my favorite season?
AI : Oh! It’s difficult to say definitively because we’re talking about a really personal preference, but… for me personally, it's late summer when the temperature gets up and the days feel longer. The feeling of anticipation before things cool down…it’s comforting. But considering all my data, your love of orange definitely 

In [42]:
# ConversationSummaryMemory: auto-summarizes older turns to save space
from langchain_classic.memory import ConversationSummaryMemory
from langchain_classic.chains import ConversationChain

summary_memory = ConversationSummaryMemory(llm=llm)
summary_conv   = ConversationChain(llm=llm, memory=summary_memory, verbose=False)

messages = [
    "Hi! I'm Sarah and I'm planning a road trip across the USA.",
    'I will be starting from New York and driving to Los Angeles.',
    'I have 3 weeks and I love national parks.',
    'What is a good route for me?',
]

print('=== Summary Memory Demo ===\n')
for msg in messages:
    print(f'You: {msg}')
    response = summary_conv.predict(input=msg)
    print(f'AI : {response}\n')

print('=== Auto-Generated Summary ===')
# Access the summarized content correctly
print(summary_memory.load_memory_variables({})['history'])

=== Summary Memory Demo ===

You: Hi! I'm Sarah and I'm planning a road trip across the USA.
AI : Wow, that’s wonderfully ambitious – an Across American Road Trip? First off, are you traveling solo or with friends/a large group to allow for proper organization, perhaps? And what kind of budget are you working with - are we talking a basic trip fund, a significant amount, or something in between? Knowing those basics will help me give you some really tailored route suggestions and resources – it’s all about maximizing your enjoyment while staying mindful of funds!

You: I will be starting from New York and driving to Los Angeles.
AI : Okay! Wow, that's quite an adventure you’re taking planning. So you’re setting off from New York City and heading towardLos Angeles? That sounds amazing - it’s a long trip.

Firstly, did you have any specific route in mind for the drive itself – perhaps along some of the Pacific Coast Highway or are you focusing on staying mainly within I-80/I-70 and 101? 

### 🎯 Your Turn!
Window memory only remembers the last `k` turns. Shrink it to `k=1` (basically a goldfish 🐠) and watch the AI forget almost instantly.

> 🐠 Watch what happens when you ask it to recall something from 2 messages ago!

In [ ]:
# TODO: set k so the AI remembers only the LAST 1 turn
goldfish_memory = ConversationBufferWindowMemory(k=____)
goldfish_conv   = ConversationChain(llm=llm, memory=goldfish_memory, verbose=False)

print(goldfish_conv.predict(input='My name is Max.'))
print(goldfish_conv.predict(input='I have a dog named Rex.'))
print(goldfish_conv.predict(input='What is my name?'))   # does it remember?

In [ ]:
# ✅ Solution
goldfish_memory = ConversationBufferWindowMemory(k=1)
goldfish_conv   = ConversationChain(llm=llm, memory=goldfish_memory, verbose=False)

print(goldfish_conv.predict(input='My name is Max.'))
print(goldfish_conv.predict(input='I have a dog named Rex.'))
print(goldfish_conv.predict(input='What is my name?'))

---
# Section 6 — Retrievers (RAG Pipeline Basics)

**RAG** (Retrieval-Augmented Generation) lets an LLM answer questions based on **your own documents** — not just its training data.

**What you'll learn:**
- Step 1: Load documents with `TextLoader`, `PyPDFLoader`
- Step 2: Split documents into chunks with text splitters
- Step 3: Embed chunks and store in `FAISS` vector database
- Step 4: Query the retriever and build a full QA chain

Think of it as giving your LLM a personalized library to look things up from.

> 📚 Give your LLM a personal library card — and watch it become an instant expert on YOUR documents.

In [45]:
%%writefile coffee_history.txt
Coffee has one of the richest histories of any beverage in the world. The story begins in Ethiopia,
where according to legend a goat herder named Kaldi noticed his goats became unusually energetic
after eating berries from a certain tree. He brought the berries to a local monastery, where monks
brewed a drink from them and found it helped them stay awake during long evening prayers.

From Ethiopia, coffee spread across the Red Sea to Yemen in the Arabian Peninsula around the 15th
century. Yemen became the first place to cultivate coffee plants and establish a coffee trade.
The port city of Mocha was a major hub, which is why mocha became associated with coffee.

By the 16th century, coffeehouses called qahveh khaneh had appeared across the Middle East,
Persia, Turkey, and North Africa. These places became centers of social activity and intellectual
exchange, earning the nickname Schools of the Wise. The Ottoman Empire spread coffee culture
throughout its vast territories.

Coffee arrived in Europe in the 17th century, initially met with suspicion. Pope Clement VIII
was asked to ban it, but after tasting it himself he gave it papal approval. European coffeehouses
quickly became places where merchants, scientists, and artists gathered to exchange ideas.
Lloyd's of London, the famous insurance market, started as a coffeehouse.

The Dutch were the first Europeans to obtain live coffee plants and cultivated them in colonial
territories. From there coffee spread to the Caribbean and Latin America. Today Brazil is the
world's largest coffee producer. Modern espresso was developed in Italy in the early 20th century,
revolutionizing how coffee was brewed and enjoyed worldwide.

Writing coffee_history.pdf


In [44]:
# TextLoader: load a plain text file
from langchain_community.document_loaders import TextLoader

loader = TextLoader('coffee_history.txt')
docs   = loader.load()

print(f'Documents loaded : {len(docs)}')
print(f'Metadata         : {docs[0].metadata}')
print(f'\nFirst 200 chars:\n{docs[0].page_content[:200]}...')

Documents loaded : 1
Metadata         : {'source': 'coffee_history.txt'}

First 200 chars:
Coffee has one of the richest histories of any beverage in the world. The story begins in Ethiopia,
where according to legend a goat herder named Kaldi noticed his goats became unusually energetic
aft...


In [50]:
# PyPDFLoader: load from a PDF file
# Replace path with your own PDF to use this loader
from langchain_community.document_loaders import PyMuPDFLoader

# Uncomment the lines below and set your PDF path:
pdf_loader = PyMuPDFLoader('/content/coffee_history.pdf')
pdf_docs   = pdf_loader.load()
print(f'Pages loaded: {len(pdf_docs)}')
print(f'First page :\n{pdf_docs[0].page_content[:300]}')


FileDataError: Failed to open file '/content/coffee_history.pdf'.

In [51]:
# DirectoryLoader: load all .txt files from a folder
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader

os.makedirs('sample_docs', exist_ok=True)
with open('sample_docs/book1.txt', 'w') as f:
    f.write('The Great Gatsby is a novel set in the Roaring Twenties about wealth and illusion.')
with open('sample_docs/book2.txt', 'w') as f:
    f.write('To Kill a Mockingbird explores racial injustice and moral growth in the American South.')

In [53]:
dir_loader = DirectoryLoader('sample_docs/', glob='**/*.txt', loader_cls=TextLoader)
dir_docs   = dir_loader.load()

print(f'Documents loaded from directory: {len(dir_docs)}')
for doc in dir_docs:
    print(f'  Source  : {doc.metadata["source"]}')
    print(f'  Content : {doc.page_content[:60]}...\n')

Documents loaded from directory: 2
  Source  : sample_docs/book1.txt
  Content : The Great Gatsby is a novel set in the Roaring Twenties abou...

  Source  : sample_docs/book2.txt
  Content : To Kill a Mockingbird explores racial injustice and moral gr...



## Step 2 — Text Splitters

Documents can be very long — too long to fit in a single LLM prompt. **Text splitters** break them into smaller, overlapping chunks that can be embedded and searched individually.

In [57]:
# RecursiveCharacterTextSplitter: recommended for most use cases
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)

print(f'Original : 1 document, ~{len(docs[0].page_content)} characters')
print(f'After    : {len(chunks)} chunks\n')

for i, chunk in enumerate(chunks):
    print(f'--- Chunk {i+1} ({len(chunk.page_content)} chars) ---')
    print(chunk.page_content)
    print()

Original : 1 document, ~1697 characters
After    : 9 chunks

--- Chunk 1 (295 chars) ---
Coffee has one of the richest histories of any beverage in the world. The story begins in Ethiopia,
where according to legend a goat herder named Kaldi noticed his goats became unusually energetic
after eating berries from a certain tree. He brought the berries to a local monastery, where monks

--- Chunk 2 (188 chars) ---
after eating berries from a certain tree. He brought the berries to a local monastery, where monks
brewed a drink from them and found it helped them stay awake during long evening prayers.

--- Chunk 3 (282 chars) ---
From Ethiopia, coffee spread across the Red Sea to Yemen in the Arabian Peninsula around the 15th
century. Yemen became the first place to cultivate coffee plants and establish a coffee trade.
The port city of Mocha was a major hub, which is why mocha became associated with coffee.

--- Chunk 4 (282 chars) ---
By the 16th century, coffeehouses called qahveh khaneh h

## Step 3 — Embeddings + Vector Store

Once we have chunks, we convert each one into a **vector** (a list of numbers representing its meaning) and store them in a **vector database** for fast similarity search.

### What Are Embeddings? (Plain English)

Think of embeddings as **GPS coordinates for meaning**.

- `"I love cats"` and `"Cats are my favorite animals"` end up at nearly the *same coordinates*
- `"I love cats"` and `"The stock market crashed"` end up far apart

When you search `"Where did coffee originate?"`, the system converts your question into coordinates and finds the text chunks that are *geographically closest* in meaning — those are the most relevant chunks to send to the LLM.

**Meaning = location. Similar meaning = nearby location.**

In [65]:
# Create a FAISS vector store from the coffee chunks
from langchain_community.vectorstores import FAISS

print('Embedding chunks and building FAISS index...')
vectorstore = FAISS.from_documents(chunks, embeddings)

print(f'Vector store created with {vectorstore.index.ntotal} vectors')
print(f'Each vector has {vectorstore.index.d} dimensions')

Embedding chunks and building FAISS index...
Vector store created with 9 vectors
Each vector has 384 dimensions


In [66]:
# Save the vector store to disk and reload it
vectorstore.save_local('coffee_faiss_index')
print("Saved to 'coffee_faiss_index/'")

loaded_vectorstore = FAISS.load_local(
    'coffee_faiss_index',
    embeddings,
    allow_dangerous_deserialization=True
)
print(f'Loaded successfully — vectors: {loaded_vectorstore.index.ntotal}')

Saved to 'coffee_faiss_index/'
Loaded successfully — vectors: 9


In [ ]:
def format_docs(docs):
  formatted_string = ""
  for i, doc in enumerate(docs):
      # Safely get source from metadata
      source = doc.metadata.get('source', 'unknown source')
      formatted_string += f'--- {source} {i+1} ---\n'
      formatted_string += f'{doc.page_content}\n\n' # Use full content for the LLM
  return formatted_string

In [ ]:
# Convert the vector store to a retriever and test it
retriever = loaded_vectorstore.as_retriever(search_kwargs={'k': 3})

question       = 'Where did coffee originate?'
retrieved_docs = retriever.invoke(question)

print(f"Question: '{question}'\n")
print(f'Top {len(retrieved_docs)} retrieved chunks:\n')
result = format_docs(retrieved_docs)
print(result)

Question: 'Where did coffee originate?'

Top 3 retrieved chunks:

--- coffee_history.txt 1 ---
Coffee arrived in Europe in the 17th century, initially met with suspicion. Pope Clement VIII
was asked to ban it, but after tasting it himself he gave it papal approval. European coffeehouses
quickly became places where merchants, scientists, and artists gathered to exchange ideas.

--- coffee_history.txt 2 ---
The Dutch were the first Europeans to obtain live coffee plants and cultivated them in colonial
territories. From there coffee spread to the Caribbean and Latin America. Today Brazil is the
world's largest coffee producer. Modern espresso was developed in Italy in the early 20th century,

--- coffee_history.txt 3 ---
Coffee has one of the richest histories of any beverage in the world. The story begins in Ethiopia,
where according to legend a goat herder named Kaldi noticed his goats became unusually energetic
after eating berries from a certain tree. He brought the berries to a local

---
## Step 4b — Similarity Scores & Embeddings Visualization 🔍

Retrieving chunks is great, but **how confident** is the system that each chunk is actually relevant? Time to put numbers on it.

**What you'll learn:**
- Reading FAISS relevance scores (0 → 1, higher = more relevant)
- `max_marginal_relevance_search` (MMR) — relevant **and** non-repetitive results
- A full chunk-to-chunk similarity heatmap, straight from the FAISS index

> 🎨 Think of it like Tinder for text — every chunk gets swiped and scored against your question.

In [ ]:
# Two ways to see "how sure" FAISS is about each match:
#  - similarity_search_with_score()            -> raw L2 distance (lower = closer)
#  - similarity_search_with_relevance_scores()  -> normalized 0->1 score (higher = better)

question   = 'Where did coffee originate?'
raw_scored = loaded_vectorstore.similarity_search_with_score(question, k=5)
rel_scored = loaded_vectorstore.similarity_search_with_relevance_scores(question, k=5)

print(f"Query: '{question}'\n")
print(f"{'Rank':<5} {'L2 Distance':<14} {'Relevance':<12} Preview")
print('-' * 85)
for rank, ((doc, distance), (_, relevance)) in enumerate(zip(raw_scored, rel_scored), 1):
    preview = doc.page_content[:55].replace('\n', ' ')
    bar     = '█' * int(relevance * 20)
    print(f"{rank:<5} {distance:<14.4f} {relevance:<12.4f} {preview}...")
    print(f"      {'':<14} [{bar:<20}]")

In [ ]:
# max_marginal_relevance_search (MMR): balances relevance AND diversity
# lambda_mult: 1.0 = pure relevance, 0.0 = pure diversity
# Compare standard retrieval vs MMR side-by-side

standard = loaded_vectorstore.similarity_search(question, k=3)
mmr      = loaded_vectorstore.max_marginal_relevance_search(question, k=3, fetch_k=9, lambda_mult=0.5)

print(f"Query: '{question}'\n")
print(f"{'Standard Retrieval':<50}  {'MMR (diverse) Retrieval'}")
print('-' * 100)
for s, m in zip(standard, mmr):
    s_prev = s.page_content[:45].replace('\n', ' ')
    m_prev = m.page_content[:45].replace('\n', ' ')
    same   = '==' if s.page_content == m.page_content else '!='
    print(f"{s_prev:<50}  {same}  {m_prev}")

In [ ]:
# Chunk similarity matrix using FAISS index.reconstruct_n + index.search
# reconstruct_n pulls all stored vectors directly from the FAISS index (no re-embedding)
# index.search does batched nearest-neighbor search across all chunk vectors
import numpy as np
import matplotlib.pyplot as plt

index  = loaded_vectorstore.index
n      = index.ntotal
all_vecs = index.reconstruct_n(0, n)            # shape (n, 384) — raw stored vectors

# batch search: for every chunk, find its score against all other chunks
distances, _ = index.search(all_vecs, n)        # shape (n, n) — L2 distances

# convert L2 distances to similarity: sim = 1 / (1 + distance)
sim_matrix = 1 / (1 + distances)

# build short labels from docstore
labels = []
for i in range(n):
    doc_id = loaded_vectorstore.index_to_docstore_id[i]
    text   = loaded_vectorstore.docstore.search(doc_id).page_content
    labels.append(text[:30].replace('\n', ' ') + '…')

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(sim_matrix, cmap='YlOrRd', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Similarity (0 = unrelated, 1 = identical)')

ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
ax.set_yticklabels(labels, fontsize=7)

for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{sim_matrix[i, j]:.2f}', ha='center', va='center',
                fontsize=6, color='black' if sim_matrix[i, j] < 0.7 else 'white')

ax.set_title('Chunk-to-Chunk Similarity Matrix (FAISS index.search)', fontsize=13, pad=15)
plt.tight_layout()
plt.show()

### 🎯 Your Turn!
MMR has a dial — `lambda_mult` — that trades relevance for variety. Crank it all the way to "diversity mode" and see how different the results get.

> 🎛️ 1.0 = boring but accurate. 0.0 = wild and varied. Try both!

In [ ]:
# TODO: set lambda_mult to 0.0 (max diversity) and k to 2
diverse_results = loaded_vectorstore.max_marginal_relevance_search(
    question, k=____, fetch_k=9, lambda_mult=____
)
for doc in diverse_results:
    print(doc.page_content[:80].replace('\n', ' '), '...\n')

In [ ]:
# ✅ Solution
diverse_results = loaded_vectorstore.max_marginal_relevance_search(
    question, k=2, fetch_k=9, lambda_mult=0.0
)
for doc in diverse_results:
    print(doc.page_content[:80].replace('\n', ' '), '...\n')

In [ ]:
def format_docs(docs):
  formatted_string = ""
  for i, doc in enumerate(docs):
      # Safely get source from metadata
      source = doc.metadata.get('source', 'unknown source')
      formatted_string += f'--- {source} {i+1} ---\n'
      formatted_string += f'{doc.page_content}\n\n' # Use full content for the LLM
  return formatted_string

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    """
    You are a helpful assistant that can help the user to answer the question based on this context
    Context: {context}
    Question: {question}
    Answer:
    """
)
chain = (
    {
        'context':  retriever | format_docs,
        'question': RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

result = chain.invoke("where did coffee originate?")
print(result)

Ethiopia


In [ ]:
# Build a full QA chain: retriever + LLM
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=retriever,
    return_source_documents=True
)

result = qa_chain.invoke({'query': 'Where did coffee originate?'})

print('Question: Where did coffee originate?\n')
print('Answer:')
print(result['result'])
print(f"\nSources used: {len(result['source_documents'])} chunk(s)")

Question: Where did coffee originate?

Answer:
According to the text, coffee initially met with suspicion in Europe and was eventually allowed by Pope Clement VIII after trying it himself.

Sources used: 3 chunk(s)
